## Data Preprocessing

This notebook extracts the raw resume text files to a combined csv file 

In [3]:
from pathlib import Path
import pandas as pd
import re

In [39]:
RAW_DIR = Path(r"data\resume_raw")

PROJECT_ROOT = RAW_DIR.parents[1]

OUTPUT_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "resume_txt_combined.csv"

print("RAW_DIR:", RAW_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_CSV:", OUTPUT_CSV)

RAW_DIR: data\resume_raw
PROJECT_ROOT: .
OUTPUT_CSV: data\resume_txt_combined.csv


In [41]:
def read_txt_file(file_path: Path) -> str:
    """
    Safely read a text file using several possible encodings.
    Returns the file content as a string.
    """
    encodings_to_try = ["utf-8", "utf-8-sig", "cp1252", "latin-1"]

    for enc in encodings_to_try:
        try:
            text = file_path.read_text(encoding=enc)
            break
        except UnicodeDecodeError:
            continue
    else:
        raise UnicodeDecodeError(
            "unknown", b"", 0, 1, f"Could not decode file: {file_path}"
        )

    # Light normalization only
    text = text.replace("\x00", " ")
    text = text.replace("\r\n", "\n").replace("\r", "\n").strip()

    return text

In [43]:
# Find only .txt files in the folder
txt_files = sorted([p for p in RAW_DIR.glob("*.txt") if p.is_file()])

print(f"Found {len(txt_files)} .txt files")

if len(txt_files) == 0:
    raise FileNotFoundError(
        f"No .txt files found in: {RAW_DIR}\n"
        "Check the folder path or make sure file extensions are .txt"
    )

Found 29783 .txt files


In [57]:
rows = []

for file_path in txt_files:
    try:
        resume_text = read_txt_file(file_path)

        row = {
            "resume_id": file_path.stem,              # e.g. 00011
            "filename": file_path.name,              # e.g. 00011.txt
            "resume_text": resume_text,
            "char_count": len(resume_text),
            "word_count": len(re.findall(r"\b\w+\b", resume_text)),
        }

        rows.append(row)

    except Exception as e:
        print(f"Skipped {file_path.name} بسبب error: {e}")

resume_df = pd.DataFrame(rows)

print("Combined shape:", resume_df.shape)
resume_df.head()

Combined shape: (29783, 5)


,resume_id,filename,resume_text,char_count,word_count
0,00001,00001.txt,"Database Administrator <span class=""hl"">Databa...",8163,1173
1,00002,00002.txt,"Database Administrator <span class=""hl"">Databa...",2185,283
2,00003,00003.txt,Oracle Database Administrator Oracle <span cla...,3805,526
3,00004,00004.txt,Amazon Redshift Administrator and ETL Develope...,3805,507
4,00005,00005.txt,Scrum Master Scrum Master Scrum Master Richmon...,4959,696


In [58]:
resume_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29783 entries, 0 to 29782
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   resume_id    29783 non-null  object
 1   filename     29783 non-null  object
 2   resume_text  29783 non-null  object
 3   char_count   29783 non-null  int64 
 4   word_count   29783 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 1.1+ MB


In [65]:
resume_df[["resume_id", "filename", "char_count", "word_count"]].head(10)

,resume_id,filename,char_count,word_count
0,00001,00001.txt,8163,1173
1,00002,00002.txt,2185,283
2,00003,00003.txt,3805,526
3,00004,00004.txt,3805,507
4,00005,00005.txt,4959,696
5,00006,00006.txt,17759,2426
6,00007,00007.txt,15283,2151
7,00008,00008.txt,4749,626
8,00009,00009.txt,5859,819
9,00010,00010.txt,6918,945


In [67]:
# Check for empty or extremely short files
resume_df[resume_df["char_count"] < 50][["resume_id", "filename", "char_count"]]

,resume_id,filename,char_count
17085,17086,17086.txt,10


In [69]:
resume_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"Saved combined CSV to:\n{OUTPUT_CSV}")

Saved combined CSV to:
data\resume_txt_combined.csv


In [70]:
sample_idx = 0
print("Filename:", resume_df.loc[sample_idx, "filename"])
print("=" * 80)
print(resume_df.loc[sample_idx, "resume_text"][:3000])  # first 3000 characters

Filename: 00001.txt
Database Administrator <span class="hl">Database</span> <span class="hl">Administrator</span> Database Administrator - Family Private Care LLC Lawrenceville, GA A self-motivated Production SQL Server Database Administrator who possesses  strong analytical and problem solving skills. My experience includes SQL Server  2005, 2008 and 2012, 2014, SSIS, as well as clustering, mirroring, and high  availability solutions in OLTP environments. I am proficient in database backup,  recovery, performance tuning, maintenance tasks, security, and consolidation.  I am confident that I would make a beneficial addition to any company. Over the  course of my career thus far, I have designed databases to fit a variety of needs,  successfully ensured the security of those databases, problem-solved in order to meet  both back-end and front-end needs, installed and tested new versions database  management systems, customized and installed applications and meticulously  monitored perfor